# IEEE 118-Bus - Pre-Processing

Builds the processed data for the SC-DCOPF using the **SCUC / StarGenLite** dataset for the
parts it is richer on, while keeping the **MATPOWER network** (tighter line limits).

**Option A mix:**
- **Generators** -> SCUC (`gen_data_scuc.csv`): 54 real units, realistic `MinProd > 0`.
  Linear cost `c1 = FuelCost * SlopeVarCost` (FuelCost = 1), intercept ignored (it belongs to UC).
- **Loads** -> SCUC hourly (`load_hourly_scuc.csv`): one hour is selected for the snapshot
  SC-DCOPF (default: peak `h021`).
- **Branches** -> MATPOWER (`branch_data_118-bus.csv`): same topology/R/X as SCUC but tighter
  ratings, kept for a richer congestion/OTS study.
- **Slack** -> bus 69 (same in both sources).

> SCUC inputs are PRELIMINARY (auto-extracted from the xlsm); validate before final use.

Output goes to `processed/scuc/`, mirroring `processed/matpower/` so the model can switch
sources by changing one variable.

## 1. Imports and configuration

In [7]:
import os
import numpy as np
import pandas as pd
import networkx as nx

DATA_DIR  = "data/scuc"       # SCUC/StarGenLite inputs (gens, hourly load)
MATPOWER_DIR = "data/matpower" # MATPOWER network (branch topology/ratings)
BASE_MVA  = 100.0
SLACK_BUS = 69          # reference bus (same as MATPOWER)
LOAD_HOUR = "h021"      # which hour of the SCUC profile to use for the snapshot (peak)
print(f"base {BASE_MVA} MVA | slack {SLACK_BUS} | load hour {LOAD_HOUR}")

base 100.0 MVA | slack 69 | load hour h021


## 2. Generators (SCUC)

All 54 SCUC units are real generators (`MaxProd > 0`) with a technical minimum
(`MinProd > 0`). For the DC model we only need `bus`, `pmin_pu`, `pmax_pu`, `c1`; the UC
fields (ramps, min up/down, startup costs) stay in the raw file for later.

In [8]:
gen_scuc = pd.read_csv(f"{DATA_DIR}/gen_data_scuc.csv")

gens = pd.DataFrame({
    "gen_id":  range(len(gen_scuc)),
    "bus":     gen_scuc["bus"].astype(int).values,
    "pmin_pu": gen_scuc["MinProd"].values / BASE_MVA,
    "pmax_pu": gen_scuc["MaxProd"].values / BASE_MVA,
    "c1":      (gen_scuc["FuelCost"] * gen_scuc["SlopeVarCost"]).values,   # $/MWh, intercept ignored
})

assert SLACK_BUS in set(gens["bus"]), "Slack bus has no generator in SCUC data!"
print(f"Generators: {len(gens)} | total capacity {gens['pmax_pu'].sum()*BASE_MVA:.0f} MW")
print(f"Generators with pmin>0: {(gens['pmin_pu']>0).sum()} / {len(gens)}")
gens.head()

Generators: 54 | total capacity 7220 MW
Generators with pmin>0: 54 / 54


,gen_id,bus,pmin_pu,pmax_pu,c1
0,0,4,0.05,0.3,27.079388
1,1,6,0.05,0.3,31.295282
2,2,8,0.05,0.3,27.899718
3,3,10,1.50,3.0,12.064470
4,4,12,1.00,3.0,11.038678


## 3. Loads

The SCUC load file has one column per hour. We take `LOAD_HOUR` as the snapshot. Buses with
no SCUC load factor get 0 (note: 8 buses that carry load in MATPOWER have no SCUC load - see
`data/README_load_sources.md`).

In [9]:
load_h = pd.read_csv(f"{DATA_DIR}/load_hourly_scuc.csv")
assert LOAD_HOUR in load_h.columns, f"{LOAD_HOUR} not found in load file."

loads = pd.DataFrame({
    "bus":   load_h["bus"].astype(int).values,
    "pd_pu": load_h[LOAD_HOUR].values,
})
print(f"Load hour {LOAD_HOUR}: total {loads['pd_pu'].sum()*BASE_MVA:.0f} MW | "
      f"buses with load {(loads['pd_pu']>0).sum()}")
loads.head()

Load hour h021: total 5592 MW | buses with load 91


,bus,pd_pu
0,1,0.810840
1,2,0.318185
2,3,0.620153
3,4,0.476998
4,5,0.000000


## 4. Branches (MATPOWER) + radial-line detection

Branches come from MATPOWER (Option A). Same bridge detection as the MATPOWER pre-processing
(MultiGraph so parallel circuits are never flagged as bridges).

In [10]:
branch_df = pd.read_csv(f"{MATPOWER_DIR}/branch_data_118-bus.csv")
br = branch_df[branch_df["status"] == 1].reset_index(drop=True).copy()

branches = pd.DataFrame({
    "branch_id": range(len(br)),
    "from_bus":  br["fbus"].astype(int).values,
    "to_bus":    br["tbus"].astype(int).values,
    "x_pu":      br["x"].values,
    "rate_pu":   br["rateA"].values / BASE_MVA,
})
branches["label"] = branches["from_bus"].astype(str) + "-" + branches["to_bus"].astype(str)

G = nx.MultiGraph(); G.add_nodes_from(range(1, 119))
for r in branches.itertuples(index=False):
    G.add_edge(r.from_bus, r.to_bus, key=r.branch_id)
bridge_pairs = set(nx.bridges(G))
branches["is_bridge"] = [((r.from_bus, r.to_bus) in bridge_pairs
                          or (r.to_bus, r.from_bus) in bridge_pairs)
                         for r in branches.itertuples(index=False)]

par_mask = branches.duplicated(subset=["from_bus", "to_bus"], keep=False)
assert not branches.loc[par_mask, "is_bridge"].any(), "A parallel line was flagged as a bridge!"
assert nx.is_connected(G), "Network is not connected."
print(f"Branches: {len(branches)} | bridges (radial): {int(branches['is_bridge'].sum())}")
branches.head()

Branches: 186 | bridges (radial): 9


,branch_id,from_bus,to_bus,x_pu,rate_pu,label,is_bridge
0,0,1,2,0.09990,1.51,1-2,False
1,1,1,3,0.04240,1.51,1-3,False
2,2,4,5,0.00798,1.76,4-5,False
3,3,3,5,0.10800,1.75,3-5,False
4,4,5,6,0.05400,1.76,5-6,False


## 5. Capacity sanity check and export

Confirm the system can serve the load, then write to `processed/scuc/` (same file names as
`processed/matpower/`).

In [11]:
total_load = loads["pd_pu"].sum() * BASE_MVA
total_pmax = gens["pmax_pu"].sum() * BASE_MVA
total_pmin = gens["pmin_pu"].sum() * BASE_MVA
print(f"Total capacity : {total_pmax:.0f} MW")
print(f"Total load     : {total_load:.0f} MW")
print(f"Total MinProd  : {total_pmin:.0f} MW")
assert total_pmax >= total_load, "Capacity below load - infeasible by construction."
assert total_pmin <= total_load, "Must-run minimum exceeds load - infeasible by construction."
print("Capacity checks passed.")

Total capacity : 7220 MW
Total load     : 5592 MW
Total MinProd  : 2134 MW
Capacity checks passed.


In [12]:
OUT_DIR = "processed/scuc"
os.makedirs(OUT_DIR, exist_ok=True)

gens.to_csv(f"{OUT_DIR}/gens.csv", index=False)
loads.to_csv(f"{OUT_DIR}/loads.csv", index=False)
branches.to_csv(f"{OUT_DIR}/branches.csv", index=False)
pd.DataFrame({
    "key":   ["base_mva", "slack_bus", "load_hour"],
    "value": [BASE_MVA,   SLACK_BUS,   LOAD_HOUR],
}).to_csv(f"{OUT_DIR}/system_meta.csv", index=False)

print(f"Wrote 4 files to {OUT_DIR}/ (gens={len(gens)}, buses={len(loads)}, branches={len(branches)})")

Wrote 4 files to processed/scuc/ (gens=54, buses=118, branches=186)
